# Week 2: measure inference rather than guess

**Question:** when batch size grows, what happens to prefill time, each user's generation speed, total output throughput, and memory use?

This notebook benchmarks a small, **random-weight** causal decoder on one NVIDIA GPU. It measures systems behavior, not language quality or production LLM performance. No results have been pre-filled.

Before running, choose **Runtime → Change runtime type → GPU**. Keep prompt length, output length, model and precision fixed during a sweep. The notebook is self-contained: it writes its own Python files and requires no model download.

The default model has about 113M parameters. Its small size may expose launch overhead. Predict the curves first; do not assume every batch-size increase is beneficial.

## 1. Record the environment

Colab normally has PyTorch installed. Use its existing build rather than replacing it during the experiment. BF16 is used when supported; otherwise this run explicitly uses FP16. All batch sizes in one sweep use the same precision.

In [ ]:
import sys, importlib.util, subprocess
import torch
from pathlib import Path

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
assert torch.cuda.is_available(), "Select an NVIDIA GPU runtime before benchmarking."
print("GPU:", torch.cuda.get_device_name(0))
precision = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
print("Precision for this entire sweep:", precision)
if importlib.util.find_spec("matplotlib") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib"], check=True)


## 2. Write the benchmark code

The next cell contains the three source files and is collapsed for readability. Expand it to inspect or modify the decoder. It uses a preallocated KV cache, writes new positions in place, and lets a one-token decode query attend to every existing key. The full prompt uses causal masking.

**Why this matters:** with one query and many keys, PyTorch's `is_causal=True` is upper-left aligned. A decode query should see all cached past/current keys, so this implementation uses `is_causal=False` for that one-token call. [SDPA documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)

In [ ]:
from pathlib import Path

SOURCES = {'model.py': '"""Small causal decoder for systems experiments, with random weights and a KV cache.\n\nThe cache is a preallocated tensor per layer, not repeated torch.cat allocations.\nNo tokenizer, downloads, dropout, sampling distribution, or serving framework.\n"""\n\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass ModelConfig:\n    layers: int = 8\n    width: int = 1024\n    heads: int = 16\n    mlp_width: int = 4096\n    vocab_size: int = 4096\n    max_sequence: int = 4096\n\n    def __post_init__(self):\n        if min(vars(self).values()) < 1:\n            raise ValueError("All model dimensions must be positive")\n        if self.width % self.heads:\n            raise ValueError("width must be divisible by heads")\n\n\nclass Attention(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.heads = config.heads\n        self.head_width = config.width // config.heads\n        self.qkv = nn.Linear(config.width, 3 * config.width, bias=False)\n        self.proj = nn.Linear(config.width, config.width, bias=False)\n\n    def forward(self, x, cache=None, start_pos=0):\n        batch, length, width = x.shape\n        q, k, v = self.qkv(x).chunk(3, dim=-1)\n        q, k, v = [t.view(batch, length, self.heads, self.head_width)\n                   .transpose(1, 2) for t in (q, k, v)]\n        if cache is not None:\n            end = start_pos + length\n            if end > cache[0].shape[2]:\n                raise ValueError("Token positions exceed the allocated KV-cache capacity")\n            cache[0][:, :, start_pos:end, :].copy_(k)\n            cache[1][:, :, start_pos:end, :].copy_(v)\n            if start_pos:\n                if length != 1:\n                    raise ValueError("Cached decode accepts exactly one new token")\n                k, v = cache[0][:, :, :end, :], cache[1][:, :, :end, :]\n        # Prefill is square causal attention. During one-token decode, the cache\n        # contains only past/current tokens, so ALL exposed keys are legal.\n        # is_causal=True with Lq=1, Lkv>1 is upper-left aligned in PyTorch;\n        # it would incorrectly let the new query see only the first key.\n        y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0,\n                                          is_causal=(start_pos == 0))\n        return self.proj(y.transpose(1, 2).contiguous().view(batch, length, width))\n\n\nclass Block(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(config.width)\n        self.attention = Attention(config)\n        self.norm2 = nn.LayerNorm(config.width)\n        self.mlp = nn.Sequential(nn.Linear(config.width, config.mlp_width, bias=False),\n                                 nn.GELU(),\n                                 nn.Linear(config.mlp_width, config.width, bias=False))\n\n    def forward(self, x, cache=None, start_pos=0):\n        x = x + self.attention(self.norm1(x), cache, start_pos)\n        return x + self.mlp(self.norm2(x))\n\n\nclass TinyDecoder(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.config = config\n        self.embedding = nn.Embedding(config.vocab_size, config.width)\n        self.position = nn.Embedding(config.max_sequence, config.width)\n        self.blocks = nn.ModuleList([Block(config) for _ in range(config.layers)])\n        self.norm = nn.LayerNorm(config.width)\n        self.lm_head = nn.Linear(config.width, config.vocab_size, bias=False)\n\n    def allocate_cache(self, batch, capacity=None):\n        capacity = self.config.max_sequence if capacity is None else capacity\n        if not 1 <= capacity <= self.config.max_sequence:\n            raise ValueError("Cache capacity must fit the model\'s position range")\n        shape = (batch, self.config.heads, capacity,\n                 self.config.width // self.config.heads)\n        weight = self.embedding.weight\n        return [(torch.empty(shape, device=weight.device, dtype=weight.dtype),\n                 torch.empty(shape, device=weight.device, dtype=weight.dtype))\n                for _ in self.blocks]\n\n    def forward(self, tokens, caches=None, start_pos=0, return_all=False):\n        length = tokens.shape[1]\n        if start_pos < 0 or start_pos + length > self.config.max_sequence:\n            raise ValueError("Token positions exceed model/cache capacity")\n        if start_pos and caches is None:\n            raise ValueError("A nonzero start_pos requires a populated KV cache")\n        if caches is not None and len(caches) != len(self.blocks):\n            raise ValueError("One KV cache pair is required per layer")\n        positions = torch.arange(start_pos, start_pos + length, device=tokens.device)\n        x = self.embedding(tokens) + self.position(positions)\n        for index, block in enumerate(self.blocks):\n            x = block(x, None if caches is None else caches[index], start_pos)\n        # A generation step needs logits only for the last input position.\n        # Every prompt position still participates in attention and fills KV.\n        x = x if return_all else x[:, -1:, :]\n        return self.lm_head(self.norm(x))\n\n\n@torch.inference_mode()\ndef check_correctness():\n    """CPU FP32: compare cached logits AND greedy generation to full recomputation."""\n    torch.manual_seed(2026)\n    config = ModelConfig(layers=2, width=32, heads=4, mlp_width=64,\n                         vocab_size=41, max_sequence=24)\n    model = TinyDecoder(config).float().eval()\n    tokens = torch.randint(config.vocab_size, (2, 13))\n    full = model(tokens, return_all=True)\n    worst = 0.0\n    # Multiple prompt lengths catch incorrect position indexing and decode masks.\n    for prompt_length in (1, 4, 9):\n        cache = model.allocate_cache(tokens.shape[0], capacity=tokens.shape[1])\n        prefill = model(tokens[:, :prompt_length], cache, return_all=True)\n        torch.testing.assert_close(prefill, full[:, :prompt_length], rtol=1e-5, atol=1e-5)\n        for pos in range(prompt_length, tokens.shape[1]):\n            cached = model(tokens[:, pos:pos + 1], cache, start_pos=pos)\n            expected = full[:, pos:pos + 1]\n            worst = max(worst, (cached - expected).abs().max().item())\n            torch.testing.assert_close(cached, expected, rtol=1e-5, atol=1e-5)\n    prompt = tokens[:, :4]\n    cache = model.allocate_cache(prompt.shape[0], capacity=prompt.shape[1] + 5)\n    next_token = model(prompt, cache).argmax(dim=-1)\n    full_sequence = prompt.clone()\n    for step in range(6):\n        expected = model(full_sequence).argmax(dim=-1)\n        torch.testing.assert_close(next_token, expected, rtol=0, atol=0)\n        full_sequence = torch.cat((full_sequence, expected), dim=1)\n        if step < 5:\n            next_token = model(next_token, cache, start_pos=prompt.shape[1] + step).argmax(dim=-1)\n    return {"device": "cpu", "dtype": "float32", "max_abs_logit_error": worst,\n            "cached_logits": "pass", "greedy_generation": "pass"}\n', 'benchmark.py': '"""Measure prefill and steady decode separately. --help needs only Python stdlib."""\n\nimport argparse\nimport csv\nfrom dataclasses import asdict\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport platform\nimport statistics\nimport time\n\n\nFIELDS = ["batch_size", "prompt_tokens", "output_tokens", "decode_steps",\n          "prefill_ms_p50", "prefill_ms_p95", "decode_step_ms_p50", "decode_step_ms_p95",\n          "per_user_tokens_s", "aggregate_output_tokens_s", "peak_allocated_gb", "status"]\n\n\ndef percentile(samples, fraction):\n    """Linearly interpolate an empirical quantile; independent of numpy/PyTorch."""\n    values = sorted(samples)\n    if not values or not 0 <= fraction <= 1:\n        raise ValueError("Need nonempty samples and a fraction in [0, 1]")\n    position = (len(values) - 1) * fraction\n    lower = int(position)\n    upper = min(lower + 1, len(values) - 1)\n    return values[lower] + (position - lower) * (values[upper] - values[lower])\n\n\ndef summarize(batch, prompt, output, prefill_ms, decode_ms, peak):\n    steps = output - 1\n    step_ms = [duration / steps for duration in decode_ms]\n    p50_step = statistics.median(step_ms)\n    return dict(zip(FIELDS, [batch, prompt, output, steps,\n                           statistics.median(prefill_ms), percentile(prefill_ms, 0.95),\n                           p50_step, percentile(step_ms, 0.95),\n                           1000 / p50_step, batch * 1000 / p50_step, peak / 1e9, "ok"]))\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--check", action="store_true", help="Run deterministic CPU FP32 correctness checks and exit")\n    parser.add_argument("--batches", type=int, nargs="+", default=[1, 2, 4, 8, 16, 32])\n    parser.add_argument("--prompt-tokens", type=int, default=512)\n    parser.add_argument("--output-tokens", type=int, default=65, help="Includes first token produced by prefill; must be >=2")\n    parser.add_argument("--precision", choices=["bf16", "fp16", "fp32"], default="bf16")\n    parser.add_argument("--warmup", type=int, default=3, help="Full prefill+decode warmups per batch size")\n    parser.add_argument("--trials", type=int, default=20)\n    parser.add_argument("--layers", type=int, default=8)\n    parser.add_argument("--width", type=int, default=1024)\n    parser.add_argument("--heads", type=int, default=16)\n    parser.add_argument("--mlp-width", type=int, default=4096)\n    parser.add_argument("--vocab-size", type=int, default=4096)\n    parser.add_argument("--max-sequence", type=int, default=4096, help="Fixed learned-position capacity; hold fixed across context-length experiments")\n    parser.add_argument("--seed", type=int, default=2026)\n    parser.add_argument("--output", type=Path, default=Path("results.csv"))\n    args = parser.parse_args()\n    if min(args.batches) < 1 or args.prompt_tokens < 1 or args.output_tokens < 2:\n        parser.error("batches/prompt must be positive; output-tokens must be >=2")\n    if args.warmup < 1 or args.trials < 3:\n        parser.error("Use >=1 warmup and >=3 measured trials (20 recommended)")\n    if len(set(args.batches)) != len(args.batches):\n        parser.error("Do not repeat batch sizes")\n    if args.prompt_tokens + args.output_tokens - 1 > args.max_sequence:\n        parser.error("prompt-tokens + output-tokens - 1 must fit max-sequence")\n    return args\n\n\ndef main():\n    args = parse_args()\n    try:\n        import torch\n        from model import ModelConfig, TinyDecoder, check_correctness\n    except ImportError as exc:\n        raise SystemExit("Install PyTorch first; see README.md. --help works without PyTorch.") from exc\n    if args.check:\n        print(json.dumps(check_correctness(), indent=2))\n        return\n    if not torch.cuda.is_available():\n        raise SystemExit("This timing experiment requires an NVIDIA CUDA GPU. CPU correctness: --check")\n    if args.precision == "bf16" and not torch.cuda.is_bf16_supported():\n        raise SystemExit("This GPU does not support BF16. Choose --precision fp16, and report that precision.")\n    torch.manual_seed(args.seed)\n    torch.cuda.manual_seed_all(args.seed)\n    # For an FP32 experiment, prevent TF32 from silently changing matrix precision.\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    dtype = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}[args.precision]\n    config = ModelConfig(layers=args.layers, width=args.width, heads=args.heads,\n                         mlp_width=args.mlp_width, vocab_size=args.vocab_size,\n                         max_sequence=args.max_sequence)\n    model = TinyDecoder(config).to(device="cuda", dtype=dtype).eval()\n    gpu = torch.cuda.get_device_properties(0)\n    metadata = {\n        "created_utc": datetime.now(timezone.utc).isoformat(),\n        "model": "TinyDecoder, random weights; no language-quality interpretation",\n        "config": asdict(config), "parameter_count": sum(p.numel() for p in model.parameters()),\n        "seed": args.seed, "precision": args.precision, "warmup": args.warmup, "trials": args.trials,\n        "prompt_tokens": args.prompt_tokens, "output_tokens": args.output_tokens,\n        "kv_cache_capacity": args.prompt_tokens + args.output_tokens - 1,\n        "batches": args.batches, "python": platform.python_version(), "pytorch": torch.__version__,\n        "cuda_runtime": torch.version.cuda, "gpu": gpu.name,\n        "gpu_total_memory_gb": gpu.total_memory / 1e9, "compute_capability": [gpu.major, gpu.minor],\n        "sdpa_backend": "PyTorch automatic selection; no backend forced", "compile": False,\n        "tf32": False, "timing": "perf_counter with CUDA synchronization before/after each phase",\n        "includes": "Python dispatch, GPU work, KV writes, greedy argmax",\n        "excludes": "model/cache allocation, tokenization, network, queueing, transfer to host",\n        "percentiles": "Across repeated trials; each decode sample is a whole decode-loop time divided by its step count. NOT serving tail latency.",\n        "memory": "PyTorch peak tensor allocation including model, preallocated full-capacity KV, inputs and temporaries; excludes driver/context and reserved unused pool",\n        "source_sha256": {name: hashlib.sha256(Path(__file__).with_name(name).read_bytes()).hexdigest()\n                          for name in ("model.py", "benchmark.py")},\n        "raw_trials": {},\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    json_path = args.output.with_suffix(".json")\n    rows = []\n\n    def save():\n        with args.output.open("w", newline="") as stream:\n            writer = csv.DictWriter(stream, fieldnames=FIELDS)\n            writer.writeheader()\n            writer.writerows(rows)\n        json_path.write_text(json.dumps(metadata, indent=2) + "\\n")\n\n    @torch.inference_mode()\n    def run_batch(batch):\n        # Allocation is outside timing but is counted in peak tensor memory.\n        prompt = torch.randint(config.vocab_size, (batch, args.prompt_tokens), device="cuda")\n        caches = model.allocate_cache(batch, args.prompt_tokens + args.output_tokens - 1)\n\n        def trial():\n            torch.cuda.synchronize()\n            start = time.perf_counter()\n            next_token = model(prompt, caches).argmax(dim=-1)\n            torch.cuda.synchronize()\n            prefill_ms = (time.perf_counter() - start) * 1000\n            # Prefill logits predict output token 1. Feed that token to obtain\n            # token 2, then continue; G output tokens require G-1 decode calls.\n            start = time.perf_counter()\n            for step in range(args.output_tokens - 1):\n                pos = args.prompt_tokens + step\n                next_token = model(next_token, caches, start_pos=pos).argmax(dim=-1)\n            torch.cuda.synchronize()\n            decode_ms = (time.perf_counter() - start) * 1000\n            return prefill_ms, decode_ms\n\n        for _ in range(args.warmup):\n            trial()\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n        prefill, decode = [], []\n        for _ in range(args.trials):\n            first, rest = trial()\n            prefill.append(first)\n            decode.append(rest)\n        peak = torch.cuda.max_memory_allocated()\n        metadata["raw_trials"][str(batch)] = {"prefill_ms": prefill, "decode_phase_ms": decode}\n        return summarize(batch, args.prompt_tokens, args.output_tokens, prefill, decode, peak)\n\n    print(f"{gpu.name} | {args.precision} | {metadata[\'parameter_count\'] / 1e6:.1f}M random parameters")\n    print("Batch  Prefill p50 ms  Decode avg-step p50 ms  Per-user tok/s  Total tok/s  Peak GB")\n    for batch in args.batches:\n        try:\n            row = run_batch(batch)\n        except torch.cuda.OutOfMemoryError:\n            row = {field: "" for field in FIELDS}\n            row.update(batch_size=batch, prompt_tokens=args.prompt_tokens,\n                       output_tokens=args.output_tokens, decode_steps=args.output_tokens - 1, status="oom")\n            print(f"{batch:5d}  OOM (recorded; no throughput fabricated)")\n        else:\n            print(f"{batch:5d}  {row[\'prefill_ms_p50\']:14.2f}  {row[\'decode_step_ms_p50\']:22.3f}"\n                  f"  {row[\'per_user_tokens_s\']:14.1f}  {row[\'aggregate_output_tokens_s\']:11.1f}"\n                  f"  {row[\'peak_allocated_gb\']:7.3f}")\n        rows.append(row)\n        save()\n        gc.collect()\n        torch.cuda.empty_cache()\n    print(f"Saved {args.output} and {json_path}")\n\n\nif __name__ == "__main__":\n    main()\n', 'plot_results.py': '"""Plot YOUR measured results.csv; no demonstration or synthetic timings included."""\n\nimport argparse\nimport csv\nimport json\nimport math\nfrom pathlib import Path\n\n\ndef load_rows(path):\n    rows = []\n    with Path(path).open(newline="") as stream:\n        for row in csv.DictReader(stream):\n            if row["status"] != "ok":\n                continue\n            values = {key: float(row[key]) for key in\n                      ("batch_size", "prefill_ms_p50", "decode_step_ms_p50",\n                       "per_user_tokens_s", "aggregate_output_tokens_s", "peak_allocated_gb")}\n            if not all(math.isfinite(value) and value > 0 for value in values.values()):\n                raise ValueError("Plot requires finite, positive measured values")\n            rows.append(values)\n    if not rows:\n        raise ValueError("No successful measurements in this CSV")\n    return sorted(rows, key=lambda row: row["batch_size"])\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("csv", type=Path)\n    parser.add_argument("--output", type=Path, default=Path("inference-batch-sweep"), help="Output filename stem")\n    args = parser.parse_args()\n    rows = load_rows(args.csv)\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    metadata_file = args.csv.with_suffix(".json")\n    metadata = json.loads(metadata_file.read_text()) if metadata_file.exists() else {}\n    plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,\n                         "figure.facecolor": "white", "axes.titlecolor": "#245875"})\n    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2))\n    batch = [row["batch_size"] for row in rows]\n    blue = "#245875"\n    for ax, metric, label, title in (\n        (axes[0, 0], "prefill_ms_p50", "Prefill + first argmax (ms)", "First-token compute time"),\n        (axes[0, 1], "decode_step_ms_p50", "Average decode-step duration (ms)", "Per-user generation latency"),\n        (axes[1, 1], "peak_allocated_gb", "Peak tensor allocation (GB)", "Memory required by the batch"),\n    ):\n        ax.plot(batch, [row[metric] for row in rows], "o-", color=blue)\n        ax.set(xlabel="Batch size", ylabel=label, title=title)\n        ax.set_xscale("log", base=2)\n        ax.set_xticks(batch, labels=[str(int(value)) for value in batch])\n        ax.grid(alpha=0.18)\n    ax = axes[1, 0]\n    ax.scatter([row["per_user_tokens_s"] for row in rows],\n               [row["aggregate_output_tokens_s"] for row in rows], color=blue, s=40)\n    for row in rows:\n        ax.annotate(f"B={int(row[\'batch_size\'])}",\n                    (row["per_user_tokens_s"], row["aggregate_output_tokens_s"]),\n                    xytext=(5, 5), textcoords="offset points", fontsize=9)\n    ax.set(xlabel="Per-user output tokens/s (steady decode)",\n           ylabel="Aggregate output tokens/s (steady decode)", title="What does a larger batch trade off?")\n    ax.grid(alpha=0.18)\n    gpu = metadata.get("gpu", "GPU metadata unavailable")\n    precision = metadata.get("precision", "unknown precision").upper()\n    fig.suptitle(f"Measured batch sweep — {gpu}, {precision}", fontsize=16, color=blue)\n    if "parameter_count" in metadata:\n        fig.text(0.5, 0.925,\n                 f"{metadata[\'parameter_count\'] / 1e6:.1f}M parameters | "\n                 f"prompt {metadata[\'prompt_tokens\']} | output {metadata[\'output_tokens\']} | "\n                 f"{metadata[\'trials\']} trials | PyTorch {metadata[\'pytorch\']}",\n                 ha="center", fontsize=10, color="#555555")\n    fig.text(0.5, 0.025,\n             "Random-weight decoder; fixed lengths. Medians across trials. Decode = whole-loop time / steps.\\n"\n             "Local synchronized wall-clock timing, not production serving latency or an H100 performance claim.",\n             ha="center", fontsize=9, color="#555555")\n    fig.tight_layout(rect=(0, 0.07, 1, 0.92))\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    for suffix in (".png", ".svg", ".pdf"):\n        output = args.output.with_suffix(suffix)\n        fig.savefig(output, dpi=180, bbox_inches="tight")\n        print(output)\n\n\nif __name__ == "__main__":\n    main()\n'}

for name, source in SOURCES.items():
    Path(name).write_text(source)
print("Wrote", ", ".join(SOURCES))


## 3. Check correctness before timing

This small deterministic CPU FP32 test compares cached logits with full causal recomputation at several positions, and checks that greedy generation returns the same token sequence. It catches position and mask errors without requiring GPU performance results.

In [ ]:
subprocess.run([sys.executable, "benchmark.py", "--check"], check=True)


## 4. Run a controlled sweep

- **Prefill** computes the prompt and chooses the first output token.
- **Decode** feeds that first token back to predict the second, and continues. Therefore, 65 output tokens require 64 decode calls after prefill.
- Timings use synchronized wall-clock measurements around whole phases, including Python dispatch, GPU work, KV writes and greedy argmax.
- Model/cache allocation, tokenization, network, queueing and transferring tokens to the host are excluded.
- Peak memory includes the model and full-capacity preallocated KV cache, plus tensor temporaries; it is not total memory reported by `nvidia-smi`.

Run the default 20 measured trials for a presentation figure. For a quick first pass, set `trials=3` and `warmup=1`; then do not treat p95 as a stable tail estimate. A batch that exceeds GPU memory is recorded as `oom`, not zero throughput.

In [ ]:
batches = [1, 2, 4, 8, 16, 32]
prompt_tokens = 512
output_tokens = 65
warmup = 3
trials = 20
output_stem = f"batch-sweep-{precision}-s{prompt_tokens}"
csv_path = Path(f"{output_stem}.csv")

command = [sys.executable, "benchmark.py",
           "--batches", *map(str, batches),
           "--prompt-tokens", str(prompt_tokens),
           "--output-tokens", str(output_tokens),
           "--precision", precision,
           "--warmup", str(warmup), "--trials", str(trials),
           "--output", str(csv_path)]
subprocess.run(command, check=True)


## 5. Inspect the measurements

`decode_step_ms_p50` is the median across trials of **decode-loop duration ÷ decode steps**. Its p95 is a percentile of these trial averages, not a per-token or production serving latency percentile.

The steady-decode rates are:

$$\text{per-user tokens/s} = 1000 / \text{median average-step ms}$$
$$\text{aggregate tokens/s} = B \times \text{per-user tokens/s}$$

These rates exclude prefill. The first-token measurement below is local first-token compute time, not network-visible time to first token.

In [ ]:
import csv, html, json
from IPython.display import display, HTML, Image
rows = list(csv.DictReader(csv_path.open()))
columns = ["batch_size", "prefill_ms_p50", "decode_step_ms_p50", "per_user_tokens_s",
           "aggregate_output_tokens_s", "peak_allocated_gb", "status"]
def readable(value):
    try:
        return f"{float(value):.3f}"
    except ValueError:
        return value
header = "".join(f"<th>{html.escape(name)}</th>" for name in columns)
body = "".join("<tr>" + "".join(f"<td>{html.escape(readable(row[name]))}</td>" for name in columns) + "</tr>" for row in rows)
display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
metadata = json.loads(csv_path.with_suffix(".json").read_text())
print("Configuration:", metadata["gpu"], metadata["precision"], metadata["config"])


In [ ]:
plot_stem = Path(f"{output_stem}-plot")
subprocess.run([sys.executable, "plot_results.py", str(csv_path), "--output", str(plot_stem)], check=True)
display(Image(filename=str(plot_stem.with_suffix(".png"))))


## 6. Explain before optimizing

1. Does total output throughput rise? Does each user's output speed rise as well?
2. Choose a per-user latency limit. Which measured batch sizes remain acceptable?
3. Where do improvements flatten? What further measurement could distinguish bandwidth, compute and launch overhead?
4. Repeat the sweep with `prompt_tokens=2048`. Why can longer context slow decode even when the model weights and KV-cache mechanism are unchanged?

A larger batch can reuse weight reads across more tokens, but every request has its own KV state. A small eager model may also be limited by launch/dispatch overhead. Hardware, dtype and automatically selected SDPA kernels affect the outcome. These plots provide evidence for a diagnosis, not proof of one bottleneck.

Save the **CSV and metadata JSON together** with the figures. For the group meeting, run this in advance and bring the real measurements; repeat live only if convenient.

In [ ]:
import zipfile
archive = Path(f"{output_stem}-results.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in [csv_path, csv_path.with_suffix(".json"),
                 *[plot_stem.with_suffix(suffix) for suffix in (".png", ".svg", ".pdf")]]:
        bundle.write(path, arcname=path.name)
print("Saved", archive)
# In Colab, download the measured data and figures:
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(archive))


## References

- [CS336 Lecture 10: inference arithmetic intensity and batching](https://github.com/stanford-cs336/lectures/blob/main/lecture_10.py)
- [CS336 Lecture 6: benchmarking and profiling](https://github.com/stanford-cs336/lectures/blob/main/lecture_06.py)
- [Berkeley Lecture 19: throughput and per-user performance](https://scalable-ai.eecs.berkeley.edu/assets/lecture_slides/lecture_19_1.pdf#page=10)
- [PyTorch CUDA synchronization](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html)
- [PyTorch peak tensor allocation](https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html)

This notebook contains no GPU benchmark outputs. Before publication, its CLI, helper arithmetic, source consistency, and CPU FP32 cache correctness were checked; no CUDA performance measurement was possible on the authoring machine.